In [1]:
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

## K-means

### Base de dados cobertura vegetal

Neste exercício você vai usar a base de dados de cobertura vegetal, explorada nos exercícios da **Parte 1 - Classificação**.

Comece carregando os dados do arquivo `cov_types.csv` armazenado na pasta do Drive.

In [6]:
base = pd.read_csv("cov_types.csv")

Exiba os dados para uma inspeção inicial.

In [7]:
base

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area,Soil_Type,Cover_Type
0,2767.0,66.0,17.0,210.0,18.0,1190.0,234.0,204.0,96.0,2251.0,2,30,Lodgepole Pine
1,2724.0,160.0,19.0,60.0,4.0,1350.0,236.0,240.0,127.0,2514.0,2,16,Lodgepole Pine
2,2360.0,65.0,7.0,127.0,21.0,1377.0,227.0,226.0,134.0,339.0,3,5,Ponderosa Pine
3,2995.0,45.0,4.0,285.0,30.0,5125.0,221.0,231.0,146.0,5706.0,0,11,Lodgepole Pine
4,2400.0,106.0,27.0,150.0,63.0,342.0,253.0,196.0,51.0,811.0,2,3,Ponderosa Pine
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2917.0,90.0,9.0,247.0,25.0,4095.0,235.0,225.0,121.0,3901.0,0,28,Lodgepole Pine
9996,3015.0,38.0,8.0,361.0,74.0,4846.0,220.0,223.0,138.0,1611.0,0,28,Lodgepole Pine
9997,3052.0,79.0,19.0,90.0,11.0,1003.0,241.0,203.0,85.0,1490.0,2,22,Spruce/Fir
9998,2958.0,58.0,6.0,319.0,19.0,2468.0,225.0,227.0,137.0,2280.0,0,28,Lodgepole Pine


Desta vez, nós vamos remover os atributos categóricos, `Wilderness_Area` e `Soil_Type`, pois para apresentá-los ao algoritmo, nós precisaríamos usar One Hot Encoding, o que adiciona muitas colunas no dataset, e como nossos dados são limitados a 10 mil exemplos, isso pode prejudicar o algoritmo K-Means.

In [8]:
base = base.drop(["Wilderness_Area", "Soil_Type"], axis=1)
base

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Cover_Type
0,2767.0,66.0,17.0,210.0,18.0,1190.0,234.0,204.0,96.0,2251.0,Lodgepole Pine
1,2724.0,160.0,19.0,60.0,4.0,1350.0,236.0,240.0,127.0,2514.0,Lodgepole Pine
2,2360.0,65.0,7.0,127.0,21.0,1377.0,227.0,226.0,134.0,339.0,Ponderosa Pine
3,2995.0,45.0,4.0,285.0,30.0,5125.0,221.0,231.0,146.0,5706.0,Lodgepole Pine
4,2400.0,106.0,27.0,150.0,63.0,342.0,253.0,196.0,51.0,811.0,Ponderosa Pine
...,...,...,...,...,...,...,...,...,...,...,...
9995,2917.0,90.0,9.0,247.0,25.0,4095.0,235.0,225.0,121.0,3901.0,Lodgepole Pine
9996,3015.0,38.0,8.0,361.0,74.0,4846.0,220.0,223.0,138.0,1611.0,Lodgepole Pine
9997,3052.0,79.0,19.0,90.0,11.0,1003.0,241.0,203.0,85.0,1490.0,Spruce/Fir
9998,2958.0,58.0,6.0,319.0,19.0,2468.0,225.0,227.0,137.0,2280.0,Lodgepole Pine


Separe a coluna `Cover_Type` em uma variável `y` (já que esta é a classe do dataset), e utilize o método `value_counts` para relembrar quantas classes esse problema contém, e qual sua frequência.

In [9]:
y = base["Cover_Type"]
y.value_counts()

Cover_Type
Lodgepole Pine       4847
Spruce/Fir           3714
Ponderosa Pine        581
Krummholz             362
Douglas-fir           278
Aspen                 163
Cottonwood/Willow      55
Name: count, dtype: int64

Agora separe o restante das colunas em uma variável `X`, e já recupere o atributo `values` para ter um NumPy array. Exiba este array.

In [10]:
X = base.iloc[:, :-1].values
X

array([[2767.,   66.,   17., ...,  204.,   96., 2251.],
       [2724.,  160.,   19., ...,  240.,  127., 2514.],
       [2360.,   65.,    7., ...,  226.,  134.,  339.],
       ...,
       [3052.,   79.,   19., ...,  203.,   85., 1490.],
       [2958.,   58.,    6., ...,  227.,  137., 2280.],
       [2682.,   91.,   13., ...,  219.,  108., 1661.]])

Como os dados estão em escalas diferentes, instancie um escalonador do tipo `StandardScaler` e escalone `X` com ele.

In [11]:
scaler = StandardScaler()
X = scaler.fit_transform(X)
X

array([[-0.71700375, -0.78934465,  0.38187624, ..., -0.96994893,
        -1.19597298,  0.19948581],
       [-0.87120088,  0.05516249,  0.6497092 , ...,  0.84361411,
        -0.38987904,  0.39804796],
       [-2.17649753, -0.79832877, -0.95728858, ...,  0.1383396 ,
        -0.20785782, -1.2440535 ],
       ...,
       [ 0.30500049, -0.67255111,  0.6497092 , ..., -1.02032568,
        -1.48200632, -0.37506096],
       [-0.03208161, -0.8612176 , -1.09120506, ...,  0.18871635,
        -0.12984873,  0.22138049],
       [-1.02181203, -0.56474169, -0.15378969, ..., -0.21429766,
        -0.88393662, -0.24595781]])

Agora instancie e ajuste um algoritmo do tipo `KMeans`. Utilize `n_clusters` igual ao número de classes do dataset.

In [12]:
km = KMeans(n_clusters=7, random_state=0)
km.fit(X)

KMeans(n_clusters=7, random_state=0)

Recupere e exiba os centroides.

In [13]:
centroids = km.cluster_centers_
centroids

array([[-0.65627708, -0.76818278,  1.34958511, -0.31456287,  0.09665821,
        -0.5085805 ,  0.57581958, -1.66382763, -1.5931611 , -0.37829236],
       [-0.20643368, -0.64761977, -0.43094942, -0.43834808, -0.49411477,
        -0.42701401,  0.55352999,  0.08830057, -0.33822162, -0.2522865 ],
       [ 0.69081218,  1.02872371,  0.00307897,  1.63524728,  1.43100492,
         0.18006833, -0.71785937,  0.71916536,  0.97658692, -0.07038585],
       [ 0.24986472, -0.44613636, -0.58382013, -0.09813798, -0.36447098,
         1.18579093,  0.42453533,  0.20453962, -0.15348504,  1.24363673],
       [ 0.29999629,  1.12902239, -0.31857831, -0.343869  , -0.39889939,
         0.12125598, -0.54024231,  0.70172304,  0.87453164, -0.28366549],
       [ 0.63513612, -0.75086602,  0.09407804,  1.35002955,  1.26291439,
        -0.20550146,  0.4913904 , -0.36128946, -0.59233874,  0.0154369 ],
       [-1.09499094,  1.3434026 ,  1.44141148, -0.29341334,  0.31594893,
        -0.56330461, -2.12684119, -0.37099061

Aplique o método `inverse_transform` do escalonador para visualizar os centroides na escala original.

In [14]:
scaler.inverse_transform(centroids)

array([[2783.93447038,   68.35547576,   24.22621185,  204.22351885,
          52.23429084, 1561.65170557,  227.86086176,  190.22621185,
          80.72531418, 1485.71992819],
       [2909.37951583,   81.77504655,   10.93035382,  178.04655493,
          17.85996276, 1689.21787709,  227.26703911,  225.00670391,
         128.98659218, 1652.61750466],
       [3159.58891753,  268.36469072,   14.17139175,  616.55154639,
         129.87371134, 2638.66623711,  193.39561856,  237.52963918,
         179.55025773, 1893.54896907],
       [3036.6246282 ,  104.20166568,    9.78881618,  249.99107674,
          25.40333135, 4211.56930399,  223.83045806,  227.31409875,
         136.09101725, 3634.00118977],
       [3050.6045082 ,  279.52868852,   11.76946721,  198.02612705,
          23.40010246, 2546.68647541,  198.12756148,  237.18340164,
         175.6255123 , 1611.05532787],
       [3144.06288032,   70.28296146,   14.85091278,  556.23630832,
         120.09330629, 2035.65314402,  225.61156187,  216

Recupere e exiba os rótulos atribuídos pelo algoritmo KMeans em uma variável `labels`.

In [15]:
labels = km.labels_
labels

array([0, 1, 1, ..., 0, 1, 1], dtype=int32)

Agora aplique o método do cotovelo para verificar se o número de clusters igual a 7 é o melhor valor para agrupar este dataset. Faça uma pesquisa com 1 até 10 clusters, e plote o gráfico correspondente.

In [16]:
wcss = []
for i in range(1, 11):
    km = KMeans(n_clusters = i)
    km.fit(X)
    wcss.append(km.inertia_)

In [17]:
wcss

[100000.00000000015,
 79588.31413040406,
 69863.66287049952,
 63192.52561006465,
 58069.47605853963,
 53055.71736717431,
 50638.558300718374,
 47873.48383272485,
 46175.283565784805,
 45176.99448345566]

In [18]:
fig = px.line(x=range(1, 11), y=wcss)
fig.show()

Qual sua conclusão a partir do gráfico?

Vamos visualizar os dados junto com os rótulos atribuídos. Para isso, precisamos utilizar a técnica PCA.

Instancie um objeto PCA com `n_components=2`, e transforme `X` para seu PCA correspondente.

In [19]:
pca =PCA(n_components=2)
X_pca = pca.fit_transform(X)

Agora utilize `X_pca` para plotar os dados, usando `labels` para definir a cor dos pontos. Desta vez, utilize `labels.astype(str)` para que a biblioteca considere que os rótulos são categóricos, e não numéricos.

In [20]:
fig = px.scatter(x=X_pca[:, 0], y=X_pca[:, 1], color=labels.astype(str))
fig.show()

Para comparar, refaça o mesmo gráfico, mas agora utilize `y` (ou seja, os rótulos originais) para definir a cor dos pontos.

In [21]:
fig = px.scatter(x=X_pca[:, 0], y=X_pca[:, 1], color=y)
fig.show()

Observe que a comparação deixa evidente que o agrupamento baseado apenas nos atributos resulta em grupos bem diferentes daqueles discriminados pelas espécies das árvores.

DICA: você pode comparar as classes originais com os rótulos gerados pelo algoritmo utilizando a função `pd.crosstab`:

In [22]:
pd.crosstab(labels, y)

Cover_Type,Aspen,Cottonwood/Willow,Douglas-fir,Krummholz,Lodgepole Pine,Ponderosa Pine,Spruce/Fir
row_0,,,,,,,
0,41,24,69,43,459,178,300
1,71,20,70,52,1467,146,860
2,9,0,2,66,373,10,316
3,3,0,0,56,913,0,708
4,26,1,17,62,898,31,917
5,7,1,2,76,417,20,463
6,6,9,118,7,320,196,150


Isso indica que nenhum rótulo é característico de apenas uma classe, e vice-versa, o que confirma a visualização do gráfico.